# GRU training: SA-AKI sequence representation

The third of four training-setup notebooks.

Input is `data/sequence_representation.npz`, the 3D array of stays by 24 hours
by 22 features, together with `data/sequence_static.parquet`, one row per stay
holding the static fields and the label.

The model has two branches. The sequence passes through GRU layers; gender and
admission type pass through a small Dense layer and are merged in only at the
final prediction layer, so they are never fed into the recurrent path.

Two imbalance-handling conditions are run as complete, independent pipelines,
each with its own Optuna search, refit and test evaluation. The train/test
split is shared between them.

- No imbalance handling: the model is trained without class weights.
- Class weighting: Keras `class_weight` computed from training class counts at
  each fit.

## Setup

Imports and file paths. The fit/transform and evaluation functions are imported
from `src/`, the same versions used by the previous two notebooks.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import optuna
import keras
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    recall_score, precision_score, f1_score,
)

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing_fit_transform import (
    winsorize_fit, winsorize_transform,
    impute_fit, impute_transform,
    scale_fit, scale_transform,
    encode,
)
from src.evaluation_utils import (
    calibration_report, plot_calibration,
    bootstrap_ci, bootstrap_metrics_report,
    subgroup_report,
    decision_curve, plot_decision_curve,
)

DATA_DIR = Path("../data")
SEQUENCE_PATH = DATA_DIR / "sequence_representation.npz"
STATIC_PATH = DATA_DIR / "sequence_static.parquet"

RANDOM_STATE = 42
N_TRIALS = 50
N_FOLDS = 5
EPOCH_CEILING = 200
EARLY_STOPPING_PATIENCE = 18
REFIT_VAL_FRACTION = 0.15

TARGET = "label"
STATIC_INPUT_COLS = ["gender_male", "admission_type_elective"]
CONDITIONS = ["no imbalance handling", "class weighting"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("keras", keras.__version__, "| optuna", optuna.__version__)

## 1. Load the data

Loads the sequence array and the static table, and checks that the shapes and
the stay ordering agree between them.

In [ ]:
bundle = np.load(SEQUENCE_PATH, allow_pickle=True)
sequences = bundle["sequences"]
sequence_stay_ids = bundle["stay_id"]
SEQ_FEATURES = [str(f) for f in bundle["feature_names"]]

static_df = pd.read_parquet(STATIC_PATH)

N_STAYS, N_HOURS, N_FEATURES = sequences.shape
print(f"sequences: {sequences.shape}  (stays x hours x features)")
print(f"static table: {static_df.shape}")
print(f"features: {len(SEQ_FEATURES)}")
print(f"\nclass balance:")
print(static_df[TARGET].value_counts().rename("count"))
print(f"positive rate: {static_df[TARGET].mean() * 100:.2f}%")
static_df.head()

In [ ]:
assert N_HOURS == 24, f"expected 24 hours per stay, got {N_HOURS}"
assert N_FEATURES == len(SEQ_FEATURES), "feature count does not match feature_names"
assert len(static_df) == N_STAYS, "static table and sequence array disagree on stay count"
assert (static_df["stay_id"].to_numpy() == sequence_stay_ids).all(), \
    "stay ordering differs between the sequence array and the static table"

missing_static = [c for c in ["stay_id", "subject_id", "admission_age",
                              "gender", "admission_type", TARGET]
                  if c not in static_df.columns]
assert not missing_static, f"static table is missing columns: {missing_static}"

print("shapes and stay ordering agree")
print(f"missing values in sequences: {int(np.isnan(sequences).sum()):,} "
      f"({np.isnan(sequences).mean() * 100:.1f}% of cells)")
print("these are the leading gaps left by preprocessing, filled by median imputation below")

## 2. Train/test split

80/20, stratified by label, `random_state=42`. The split is computed once as
row indices and applied to both the sequence array and the static table, so
each stay's sequence and static row stay on the same side.

In [ ]:
labels = static_df[TARGET].to_numpy()

train_idx, test_idx = train_test_split(
    np.arange(N_STAYS), test_size=0.20, stratify=labels, random_state=RANDOM_STATE
)

seq_train, seq_test = sequences[train_idx], sequences[test_idx]
static_train = static_df.iloc[train_idx].reset_index(drop=True)
static_test = static_df.iloc[test_idx].reset_index(drop=True)

print(f"train: {len(train_idx):,} stays, positive rate {static_train[TARGET].mean() * 100:.2f}%")
print(f"test:  {len(test_idx):,} stays, positive rate {static_test[TARGET].mean() * 100:.2f}%")
print(f"\npatient overlap between train and test: "
      f"{len(set(static_train['subject_id']) & set(static_test['subject_id']))}")

## 3. Fold processing and class weights

The sequence array is 3D and the shared fit/transform functions take flat
tables, so each array is reshaped to `(stays * hours, features)`, passed
through Winsorize, imputation and scaling exactly as in the previous two
notebooks, then reshaped back. Fitting uses the training portion only.

`encode()` is applied to the static table alone. The sequence array never goes
through it.

Class weights follow `n_samples / (n_classes * n_samples_in_class)` and are
recomputed wherever they are used.

In [ ]:
def process_sequences(fit_seq, apply_seq, feature_cols=None):
    """Fit Winsorize/impute/scale on fit_seq, apply to both 3D arrays.

    Returns (fit_seq_processed, apply_seq_processed, fitted_params).
    """
    feature_cols = feature_cols or SEQ_FEATURES
    fit_shape, apply_shape = fit_seq.shape, apply_seq.shape
    n_features = fit_shape[-1]

    fit_flat = pd.DataFrame(fit_seq.reshape(-1, n_features), columns=feature_cols)
    apply_flat = pd.DataFrame(apply_seq.reshape(-1, n_features), columns=feature_cols)

    bounds = winsorize_fit(fit_flat, feature_cols)
    fit_flat = winsorize_transform(fit_flat, bounds)
    apply_flat = winsorize_transform(apply_flat, bounds)

    medians = impute_fit(fit_flat, feature_cols)
    fit_flat = impute_transform(fit_flat, medians)
    apply_flat = impute_transform(apply_flat, medians)

    scaler = scale_fit(fit_flat, feature_cols)
    fit_flat = scale_transform(fit_flat, scaler, feature_cols)
    apply_flat = scale_transform(apply_flat, scaler, feature_cols)

    return (fit_flat.to_numpy(dtype="float32").reshape(fit_shape),
            apply_flat.to_numpy(dtype="float32").reshape(apply_shape),
            {"bounds": bounds, "medians": medians, "scaler": scaler})


def process_static(fit_df, apply_df):
    """Apply encode() to both static tables and return the model input arrays."""
    fit_encoded = encode(fit_df)[STATIC_INPUT_COLS].to_numpy(dtype="float32")
    apply_encoded = encode(apply_df)[STATIC_INPUT_COLS].to_numpy(dtype="float32")
    return fit_encoded, apply_encoded


def class_weights(y):
    """Return n_samples / (n_classes * n_samples_in_class) for each class."""
    counts = np.bincount(y.astype(int), minlength=2)
    n_samples, n_classes = len(y), 2
    return {c: float(n_samples / (n_classes * counts[c])) for c in (0, 1)}

## 4. Model architecture

Built with the functional API, since the model takes two inputs.

The sequence input of shape (24, 22) passes through 1 or 2 GRU layers. Where
there are two, the first returns sequences and the second returns only the
final state; this follows from the layer count rather than being tuned. The
static input of shape (2,) passes through one Dense layer. The two branch
outputs are concatenated and passed to a single sigmoid unit.

GRU internal activation is tanh, its standard default. Dense layers use ReLU.
Optimizer Adam, loss binary crossentropy, with AUPRC tracked so early stopping
can monitor it.

In [ ]:
def build_model(n_hours, n_features, n_static, n_gru_layers, gru_units,
                dropout, recurrent_dropout, static_units, learning_rate):
    """Build and compile the two-branch GRU model. Returns the compiled model."""
    sequence_input = keras.Input(shape=(n_hours, n_features), name="sequence")

    x = sequence_input
    for layer in range(n_gru_layers):
        # every layer except the last returns sequences, so the next GRU has
        # a sequence to consume; the last returns only the final state
        x = keras.layers.GRU(
            gru_units,
            return_sequences=(layer < n_gru_layers - 1),
            dropout=dropout,
            recurrent_dropout=recurrent_dropout,
        )(x)

    static_input = keras.Input(shape=(n_static,), name="static")
    s = keras.layers.Dense(static_units, activation="relu")(static_input)

    merged = keras.layers.Concatenate()([x, s])
    output = keras.layers.Dense(1, activation="sigmoid")(merged)

    model = keras.Model(inputs=[sequence_input, static_input], outputs=output,
                        name="gru_dual_branch")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(curve="PR", name="auprc")],
    )
    return model


def early_stopping():
    """Return an EarlyStopping callback monitoring validation AUPRC."""
    return keras.callbacks.EarlyStopping(
        monitor="val_auprc", mode="max",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )

In [ ]:
# shape check on an untrained model, before any search runs
_demo = build_model(N_HOURS, N_FEATURES, len(STATIC_INPUT_COLS),
                    n_gru_layers=2, gru_units=32, dropout=0.2,
                    recurrent_dropout=0.1, static_units=8, learning_rate=1e-3)
_demo.summary()
keras.backend.clear_session()

## 5. Optuna objective and search space

Hyperparameters are selected on inner folds and evaluated on a test set held
out of selection (Cawley & Talbot, 2010; Varma & Simon, 2006).

Each trial builds five stratified folds from the training split. The sequence
transforms and the static encoding are fitted on each fold's own training
portion. For class weighting, the weight dictionary is recomputed from that
fold's training labels. Training runs to a ceiling of 200 epochs with early
stopping on the validation fold, monitoring AUPRC.

Mean fold AUPRC is reported after each fold, allowing a trial to be pruned
before all five folds run. The epoch reaching the best validation AUPRC is
stored on the trial for use at refit.

| Parameter | Status | Range / value |
|---|---|---|
| Number of GRU layers | Tuned | 1 to 2, integer |
| Units per GRU layer | Tuned | 8 to 64, integer, log scale |
| Dropout | Tuned | 0.0 to 0.5 |
| Recurrent dropout | Tuned | 0.0 to 0.3 |
| Static-branch units | Tuned | 4 to 16, integer |
| Learning rate | Tuned | 1e-4 to 1e-2, log scale |
| Batch size | Tuned | 16, 32, 64 or 128, categorical |
| Return sequences | Fixed (structural) | Set by GRU layer count |
| Epochs | Fixed | Ceiling 200, early stopping |
| L2 weight regularization | Fixed | 0 |
| Activation | Fixed | tanh (GRU internal), ReLU (dense) |
| Optimizer | Fixed | Adam |
| `class_weight` | Fixed | Computed from class counts, class weighting only |

In [ ]:
def suggest_params(trial):
    return {
        "n_gru_layers": trial.suggest_int("n_gru_layers", 1, 2),
        "gru_units": trial.suggest_int("gru_units", 8, 64, log=True),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "recurrent_dropout": trial.suggest_float("recurrent_dropout", 0.0, 0.3),
        "static_units": trial.suggest_int("static_units", 4, 16),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64, 128]),
    }


def make_objective(condition):
    """Return the Optuna objective for the given imbalance-handling condition."""

    def objective(trial):
        params = suggest_params(trial)
        batch_size = params.pop("batch_size")
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

        fold_scores, fold_best_epochs = [], []
        y_train_all = static_train[TARGET].to_numpy()

        for fold, (tr_idx, va_idx) in enumerate(skf.split(seq_train, y_train_all)):
            seq_fit, seq_val, _ = process_sequences(seq_train[tr_idx], seq_train[va_idx])
            static_fit, static_val = process_static(
                static_train.iloc[tr_idx].reset_index(drop=True),
                static_train.iloc[va_idx].reset_index(drop=True),
            )
            y_fit = y_train_all[tr_idx].astype("float32")
            y_val = y_train_all[va_idx].astype("float32")

            weights = class_weights(y_fit) if condition == "class weighting" else None

            keras.utils.set_random_seed(RANDOM_STATE)
            model = build_model(N_HOURS, N_FEATURES, len(STATIC_INPUT_COLS), **params)
            history = model.fit(
                [seq_fit, static_fit], y_fit,
                validation_data=([seq_val, static_val], y_val),
                epochs=EPOCH_CEILING,
                batch_size=batch_size,
                class_weight=weights,
                callbacks=[early_stopping()],
                verbose=0,
            )

            proba = model.predict([seq_val, static_val], verbose=0).ravel()
            fold_scores.append(average_precision_score(y_val, proba))
            fold_best_epochs.append(int(np.argmax(history.history["val_auprc"])) + 1)

            keras.backend.clear_session()

            trial.report(float(np.mean(fold_scores)), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("mean_best_epoch", int(np.mean(fold_best_epochs)))
        return float(np.mean(fold_scores))

    return objective

## 6. Run the Optuna study, once per condition

50 trials per condition, maximising mean fold AUPRC, with median pruning
(Akiba et al., 2019). The two studies are independent.

In [ ]:
def run_study(condition):
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        study_name=f"gru_{condition.lower().replace(' ', '_')}",
    )
    study.optimize(make_objective(condition), n_trials=N_TRIALS, show_progress_bar=True)

    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"\n{condition}: {len(study.trials)} trials, {pruned} pruned")
    print(f"best mean CV AUPRC: {study.best_value:.4f}")
    print(f"best epoch (mean over folds): {study.best_trial.user_attrs['mean_best_epoch']}")
    print("best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    return study

### No imbalance handling

In [ ]:
study_no_weighting = run_study("no imbalance handling")

### Class weighting

In [ ]:
study_class_weighting = run_study("class weighting")

## 7. Refit on the full training split

For each condition, the sequence transforms and the static encoding are fitted
once on the entire 80% training split. One model per condition is then trained
on that processed data using the condition's winning hyperparameters.

Early stopping needs a validation set, so a stratified 15% of the training
split is held back for it. The class weight dictionary is computed from the
full training split. The test set takes no part in this step.

In [ ]:
def refit(study, condition):
    params = dict(study.best_params)
    batch_size = params.pop("batch_size")

    seq_train_proc, seq_test_proc, fitted = process_sequences(seq_train, seq_test)
    static_train_proc, static_test_proc = process_static(static_train, static_test)
    y_train_all = static_train[TARGET].to_numpy().astype("float32")

    fit_rows, val_rows = train_test_split(
        np.arange(len(y_train_all)), test_size=REFIT_VAL_FRACTION,
        stratify=y_train_all, random_state=RANDOM_STATE,
    )

    weights = class_weights(y_train_all) if condition == "class weighting" else None

    keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(N_HOURS, N_FEATURES, len(STATIC_INPUT_COLS), **params)
    history = model.fit(
        [seq_train_proc[fit_rows], static_train_proc[fit_rows]], y_train_all[fit_rows],
        validation_data=([seq_train_proc[val_rows], static_train_proc[val_rows]],
                         y_train_all[val_rows]),
        epochs=EPOCH_CEILING,
        batch_size=batch_size,
        class_weight=weights,
        callbacks=[early_stopping()],
        verbose=0,
    )

    best_epoch = int(np.argmax(history.history["val_auprc"])) + 1
    print(f"{condition}: refit on {len(fit_rows):,} stays "
          f"({len(val_rows):,} held back for early stopping), best epoch {best_epoch}"
          + (f", class weights {weights}" if weights else ""))
    return model, seq_test_proc, static_test_proc, fitted, best_epoch


(model_no_weighting, seq_test_no_weighting, static_test_no_weighting,
 fitted_no_weighting, epoch_no_weighting) = refit(study_no_weighting, "no imbalance handling")

(model_class_weighting, seq_test_class_weighting, static_test_class_weighting,
 fitted_class_weighting, epoch_class_weighting) = refit(study_class_weighting, "class weighting")

## 8. Evaluate on the held-out test set

The transform parameters fitted in step 7 are applied to the test set
unchanged. Nothing is refitted on test. AUPRC is the primary metric, with
AUROC, recall, specificity, precision and F1 reported at a 0.5 decision
threshold.

In [ ]:
y_test = static_test[TARGET].to_numpy().astype("float32")


def evaluate(model, seq_test_proc, static_test_proc, condition):
    proba = model.predict([seq_test_proc, static_test_proc], verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)

    return proba, {
        "condition": condition,
        "AUPRC": average_precision_score(y_test, proba),
        "AUROC": roc_auc_score(y_test, proba),
        "recall_AKI": recall_score(y_test, pred),                    # positive class
        "specificity_non_AKI": recall_score(y_test, pred, pos_label=0),  # negative class
        "precision": precision_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    }


proba_no_weighting, metrics_no_weighting = evaluate(
    model_no_weighting, seq_test_no_weighting, static_test_no_weighting,
    "no imbalance handling")
proba_class_weighting, metrics_class_weighting = evaluate(
    model_class_weighting, seq_test_class_weighting, static_test_class_weighting,
    "class weighting")

print(f"test stays: {len(y_test):,}")
print(f"test positive rate: {y_test.mean() * 100:.2f}%  "
      f"(AUPRC of a random baseline equals the positive rate)")

## 9. Summary table

One row per condition.

In [ ]:
summary = (
    pd.DataFrame([metrics_no_weighting, metrics_class_weighting])
      .set_index("condition")
      .round(4)
)
summary

## 10. Calibration of the predicted probabilities

Calibration compares predicted probabilities against the rates observed at
those probabilities, where AUPRC and AUROC measure ranking.

Predictions come from the models refitted in step 7. Nothing is refitted,
retrained or resampled here. Both conditions are drawn on one chart.

In [ ]:
bins_no_weighting = calibration_report(y_test, proba_no_weighting,
                                       label="no imbalance handling")
bins_class_weighting = calibration_report(y_test, proba_class_weighting,
                                          label="class weighting")

fig_calibration = plot_calibration([(bins_no_weighting, "no imbalance handling"),
                                    (bins_class_weighting, "class weighting")])

## 11. Bootstrap confidence intervals

The test set is resampled with replacement and each metric from step 8 is
recomputed on every resample, giving an interval around each point estimate.

Predictions come from the models refitted in step 7. Only the test set rows are
resampled.

In [ ]:
from IPython.display import display

boot_no_weighting = bootstrap_metrics_report(y_test, proba_no_weighting, threshold=0.5,
                                             label="no imbalance handling")
boot_class_weighting = bootstrap_metrics_report(y_test, proba_class_weighting, threshold=0.5,
                                                label="class weighting")

display(boot_no_weighting)
display(boot_class_weighting)

## 12. Subgroup performance

Performance is reported separately by gender, age band and admission type, each
with the bootstrap intervals used in step 11. Group sizes are reported
alongside, and subgroup intervals are wider than the overall result.

Subgroup values come from the static table, not the sequence array. Predictions
come from the models refitted in step 7. Nothing is refitted or retrained.

In [ ]:
gender_values = static_test["gender"]

age_bands = pd.cut(static_test["admission_age"], bins=[18, 40, 65, 90],
                   labels=["18-40", "40-65", "65-90"], include_lowest=True)

# the encoded emergency/elective split; several raw admission_type categories
# are too small to bootstrap
admission_groups = pd.Series(
    static_test_no_weighting[:, STATIC_INPUT_COLS.index("admission_type_elective")]
).map({0.0: "Emergency", 1.0: "Elective"})

print("group sizes")
for name, values in [("gender", gender_values), ("age band", age_bands),
                     ("admission type", admission_groups)]:
    counts = values.value_counts(dropna=False).sort_index().to_dict()
    print(f"  {name}: {counts}")
print()

In [ ]:
SUBGROUPS = {
    "Gender": gender_values,
    "Age band": age_bands,
    "Admission type": admission_groups,
}
CONDITION_PROBA = {
    "no imbalance handling": proba_no_weighting,
    "class weighting": proba_class_weighting,
}

subgroup_tables = {}
for attribute, values in SUBGROUPS.items():
    for condition, proba in CONDITION_PROBA.items():
        key = f"{condition} -- {attribute}"
        subgroup_tables[key] = subgroup_report(y_test, proba, values, label=key)

### Subgroup results

One table per condition and attribute, each row a metric within one subgroup,
with the point estimate, the confidence interval and the subgroup size.

In [ ]:
for key, table in subgroup_tables.items():
    print(key)
    display(table)

## 13. Decision curve analysis

Net benefit is computed across the full range of thresholds and compared
against two reference strategies, flagging every patient and flagging none
(Vickers & Elkin, 2006). A model is ahead over the range where its curve sits
above both reference lines. The 0.4 to 0.7 threshold range is printed
alongside the chart.

Predictions come from the models refitted in step 7. Nothing is refitted or
retrained.

In [ ]:
curve_no_weighting = decision_curve(y_test, proba_no_weighting)
curve_class_weighting = decision_curve(y_test, proba_class_weighting)

fig_dca = plot_decision_curve([(curve_no_weighting, "no imbalance handling"),
                               (curve_class_weighting, "class weighting")])

print(curve_no_weighting[(curve_no_weighting["threshold"] >= 0.4) & (curve_no_weighting["threshold"] <= 0.7)].round(4).to_string(index=False))
print()
print(curve_class_weighting[(curve_class_weighting["threshold"] >= 0.4) & (curve_class_weighting["threshold"] <= 0.7)].round(4).to_string(index=False))

## Output

Per condition: winning hyperparameters, test metrics, and the fitted Winsorize
bounds, medians and scaler parameters. Written to `data/`, which is gitignored.
The scaler is stored as its centre and scale arrays rather than the object
itself.

In [ ]:
def save_results(study, metrics, fitted, best_epoch, condition):
    payload = {
        "condition": condition,
        "best_params": study.best_params,
        "epoch_used_at_refit": best_epoch,
        "best_cv_auprc": study.best_value,
        "test_metrics": {k: v for k, v in metrics.items() if k != "condition"},
        "fitted_params": {
            "winsorize_bounds": {k: [float(lo), float(hi)] for k, (lo, hi) in fitted["bounds"].items()},
            "medians": {k: float(v) for k, v in fitted["medians"].items()},
            "scaler_center": [float(v) for v in fitted["scaler"].center_],
            "scaler_scale": [float(v) for v in fitted["scaler"].scale_],
            "scaler_feature_order": SEQ_FEATURES,
        },
    }
    path = DATA_DIR / f"gru_{condition.lower().replace(' ', '_')}_results.json"
    path.write_text(json.dumps(payload, indent=2))
    print(f"wrote {path.name}  ({path.stat().st_size / 1024:.1f} KB)")


save_results(study_no_weighting, metrics_no_weighting, fitted_no_weighting,
             epoch_no_weighting, "no imbalance handling")
save_results(study_class_weighting, metrics_class_weighting, fitted_class_weighting,
             epoch_class_weighting, "class weighting")